# European Unemployment Analysis (2015-2024)
## Comprehensive Data Science Project

**Dataset:** European Unemployment Rates by Country, Age Group, and Gender

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Loading and Initial Exploration

In [ ]:
df = pd.read_csv('disoccupazione.csv')
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Info:")
df.info()
print("\nBasic Statistics:")
df.describe()

## 3. Data Cleaning and Preprocessing

In [ ]:
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Percentage of missing data: {(df.isnull().sum().sum() / df.size) * 100:.2f}%")

In [ ]:
year_cols = [str(year) for year in range(2015, 2025)]
df_clean = df.copy()
print(f"Unique countries: {df_clean['ISO'].nunique()}")
print(f"Unique age groups: {df_clean['AGE'].nunique()}")
print(f"Gender distribution:\n{df_clean['SEX'].value_counts()}")

## 4. Exploratory Data Analysis (EDA)
### Visualization 1: Missing Data Heatmap

In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(df[year_cols].isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Pattern Across Years', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Records', fontsize=12)
plt.tight_layout()
plt.show()

### Visualization 2: Unemployment Trends Over Time by Gender

In [ ]:
df_melted = df.melt(id_vars=['SEX', 'AGE', 'ISO'], value_vars=year_cols, 
                    var_name='Year', value_name='Unemployment_Rate')
df_melted['Year'] = df_melted['Year'].astype(int)

avg_by_gender_year = df_melted.groupby(['Year', 'SEX'])['Unemployment_Rate'].mean().reset_index()

plt.figure(figsize=(14, 6))
for sex in ['F', 'M']:
    data = avg_by_gender_year[avg_by_gender_year['SEX'] == sex]
    plt.plot(data['Year'], data['Unemployment_Rate'], marker='o', linewidth=2.5, 
             label=f"{'Female' if sex == 'F' else 'Male'}", markersize=8)

plt.title('Average Unemployment Rate Trends by Gender (2015-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization 3: Top 10 Countries with Highest Unemployment (2024)

In [ ]:
top_countries_2024 = df.groupby('ISO')['2024'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 7))
colors = sns.color_palette('rocket', len(top_countries_2024))
bars = plt.barh(range(len(top_countries_2024)), top_countries_2024.values, color=colors)
plt.yticks(range(len(top_countries_2024)), top_countries_2024.index)
plt.xlabel('Average Unemployment Rate (%)', fontsize=12)
plt.title('Top 10 Countries with Highest Unemployment Rate (2024)', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()

for i, v in enumerate(top_countries_2024.values):
    plt.text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### Visualization 4: Age Group Distribution Analysis

In [ ]:
age_unemployment = df_melted.groupby('AGE')['Unemployment_Rate'].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(14, 7))
sns.barplot(x=age_unemployment.values, y=age_unemployment.index, palette='mako')
plt.xlabel('Average Unemployment Rate (%)', fontsize=12)
plt.ylabel('Age Group', fontsize=12)
plt.title('Average Unemployment Rate by Age Group (Top 15)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Visualization 5: Correlation Heatmap Between Years

In [ ]:
correlation_matrix = df[year_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Correlation Between Unemployment Rates Across Years', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Visualization 6: Distribution of Unemployment Rates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(df['2024'].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Histogram: 2024 Unemployment Rate', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Unemployment Rate (%)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].boxplot([df[year].dropna() for year in year_cols], labels=year_cols)
axes[0, 1].set_title('Boxplot: Unemployment Rate Distribution by Year', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Unemployment Rate (%)')
axes[0, 1].tick_params(axis='x', rotation=45)

stats.probplot(df['2024'].dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot: 2024 Unemployment Rate', fontsize=12, fontweight='bold')

df['2024'].dropna().plot(kind='kde', ax=axes[1, 1], color='darkgreen', linewidth=2)
axes[1, 1].set_title('Density Plot: 2024 Unemployment Rate', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Unemployment Rate (%)')
axes[1, 1].set_ylabel('Density')

plt.tight_layout()
plt.show()

## 5. Statistical Analysis

In [ ]:
male_data = df[df['SEX'] == 'M']['2024'].dropna()
female_data = df[df['SEX'] == 'F']['2024'].dropna()

t_stat, p_value = stats.ttest_ind(male_data, female_data)
print(f"T-Test: Gender Unemployment Comparison (2024)")
print(f"Male Mean: {male_data.mean():.2f}%")
print(f"Female Mean: {female_data.mean():.2f}%")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

### Visualization 7: Gender Comparison Violin Plot

In [ ]:
plt.figure(figsize=(12, 6))
df_2024 = df[['SEX', '2024']].dropna()
df_2024['Gender'] = df_2024['SEX'].map({'F': 'Female', 'M': 'Male'})
sns.violinplot(data=df_2024, x='Gender', y='2024', palette='Set2')
plt.title('Unemployment Rate Distribution by Gender (2024)', fontsize=16, fontweight='bold')
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.xlabel('Gender', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Time Series Analysis
### Visualization 8: Country-wise Trends (Selected Countries)

In [ ]:
selected_countries = ['IT', 'ES', 'GR', 'DE', 'FR', 'SE']
country_trends = df[df['ISO'].isin(selected_countries)].groupby('ISO')[year_cols].mean()

plt.figure(figsize=(14, 7))
for country in selected_countries:
    plt.plot(year_cols, country_trends.loc[country], marker='o', linewidth=2, label=country)

plt.title('Unemployment Trends: Selected European Countries (2015-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average Unemployment Rate (%)', fontsize=12)
plt.legend(title='Country', fontsize=10)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Machine Learning: Clustering Analysis
### Visualization 9: K-Means Clustering of Countries

In [ ]:
country_avg = df.groupby('ISO')[year_cols].mean().dropna()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(country_avg)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_data)

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

plt.figure(figsize=(14, 8))
scatter = plt.scatter(pca_data[:, 0], pca_data[:, 1], c=clusters, cmap='viridis', 
                     s=200, alpha=0.6, edgecolors='black', linewidth=1.5)

for i, country in enumerate(country_avg.index):
    plt.annotate(country, (pca_data[i, 0], pca_data[i, 1]), fontsize=9, fontweight='bold')

plt.colorbar(scatter, label='Cluster')
plt.title('K-Means Clustering of Countries Based on Unemployment Patterns', fontsize=16, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.2f}%")

## 8. Predictive Modeling
### Random Forest Regression for 2024 Prediction

In [ ]:
df_model = df[['SEX', 'AGE', 'ISO'] + year_cols].copy()
df_model = df_model.dropna(subset=['2024'])

df_model['SEX_encoded'] = df_model['SEX'].map({'F': 0, 'M': 1})
df_model['ISO_encoded'] = pd.factorize(df_model['ISO'])[0]
df_model['AGE_encoded'] = pd.factorize(df_model['AGE'])[0]

feature_cols = ['SEX_encoded', 'ISO_encoded', 'AGE_encoded'] + [str(y) for y in range(2015, 2024)]
X = df_model[feature_cols].dropna()
y = df_model.loc[X.index, '2024']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Random Forest Model Performance:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {np.sqrt(mse):.4f}")

### Visualization 10: Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis')
plt.title('Top 10 Feature Importance for Unemployment Prediction', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

### Visualization 11: Actual vs Predicted Values

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Unemployment Rate (%)', fontsize=12)
plt.ylabel('Predicted Unemployment Rate (%)', fontsize=12)
plt.title('Actual vs Predicted Unemployment Rates (2024)', fontsize=16, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Advanced Analysis: Youth vs Adult Unemployment
### Visualization 12: Youth (15-24) vs Adult (25-64) Comparison

In [ ]:
youth_data = df[df['AGE'] == '15-24'].groupby('ISO')['2024'].mean().dropna()
adult_data = df[df['AGE'] == '25-64'].groupby('ISO')['2024'].mean().dropna()

common_countries = youth_data.index.intersection(adult_data.index)
comparison_df = pd.DataFrame({
    'Youth (15-24)': youth_data[common_countries],
    'Adult (25-64)': adult_data[common_countries]
}).sort_values('Youth (15-24)', ascending=False).head(15)

comparison_df.plot(kind='barh', figsize=(12, 8), color=['#FF6B6B', '#4ECDC4'])
plt.title('Youth vs Adult Unemployment Rates by Country (2024) - Top 15', fontsize=16, fontweight='bold')
plt.xlabel('Unemployment Rate (%)', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.legend(title='Age Group', fontsize=10)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Key Insights and Summary Statistics

In [ ]:
print("="*60)
print("KEY INSIGHTS FROM THE ANALYSIS")
print("="*60)

print("\n1. OVERALL TRENDS:")
print(f"   - Average unemployment 2015: {df['2015'].mean():.2f}%")
print(f"   - Average unemployment 2024: {df['2024'].mean():.2f}%")
print(f"   - Change: {df['2024'].mean() - df['2015'].mean():.2f} percentage points")

print("\n2. GENDER ANALYSIS:")
print(f"   - Female unemployment (2024): {df[df['SEX']=='F']['2024'].mean():.2f}%")
print(f"   - Male unemployment (2024): {df[df['SEX']=='M']['2024'].mean():.2f}%")

print("\n3. AGE GROUP INSIGHTS:")
youth_avg = df[df['AGE']=='15-24']['2024'].mean()
adult_avg = df[df['AGE']=='25-64']['2024'].mean()
print(f"   - Youth (15-24) unemployment: {youth_avg:.2f}%")
print(f"   - Adult (25-64) unemployment: {adult_avg:.2f}%")
print(f"   - Youth unemployment is {youth_avg/adult_avg:.1f}x higher than adults")

print("\n4. COUNTRY EXTREMES (2024):")
country_avg_2024 = df.groupby('ISO')['2024'].mean().dropna()
print(f"   - Highest: {country_avg_2024.idxmax()} ({country_avg_2024.max():.2f}%)")
print(f"   - Lowest: {country_avg_2024.idxmin()} ({country_avg_2024.min():.2f}%)")

print("\n5. MODEL PERFORMANCE:")
print(f"   - R² Score: {r2:.4f}")
print(f"   - Prediction accuracy: {r2*100:.2f}%")

print("\n" + "="*60)